# EHR + Wearable Fusion (Final QC180 Pipeline)

This notebook is the cleaned final pipeline extracted from the original exploratory notebook.

Run cells in order:
1. Model definitions
2. Core helpers + checkpoint loading
3. QC residual fusion trainer (`run_one_disease_qc`)
4. Build QC180 wearable snapshot features for 10 outcomes
5. Leakage check
6. Train/evaluate 10 outcomes and export summary CSVs


In [ ]:
# -------------------------
# 2) Model defs
# -------------------------
class EHRTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, max_days=1095, hidden_dim=256, n_layers=4, n_heads=8):
        super().__init__()
        self.max_days = max_days
        self.concept_embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.time_embedding = nn.Embedding(max_days + 1, hidden_dim, padding_idx=0)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.2)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=0.2,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def encode(self, input_ids, time_ids, attention_mask):
        t_ids = torch.clamp(time_ids, max=self.max_days)
        x = self.concept_embedding(input_ids) + self.time_embedding(t_ids)
        x = self.layer_norm(x)
        x = self.dropout(x)
        padding_mask = (attention_mask == 0)
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        return x[:, 0, :]

class MultiDiseaseClassifier(nn.Module):
    def __init__(self, shared_encoder, disease_list, encoder_out_dim=256, mlp_hidden_dim=64):
        super().__init__()
        self.encoder = shared_encoder
        self.disease_heads = nn.ModuleDict({
            d: nn.Sequential(
                nn.Linear(encoder_out_dim, mlp_hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(mlp_hidden_dim, 1)
            ) for d in disease_list
        })
        self.target_disease = None

    def set_target_disease(self, target_disease):
        self.target_disease = target_disease

    def forward(self, input_ids, time_ids, attention_mask):
        feat = self.encoder.encode(input_ids, time_ids, attention_mask)
        if self.target_disease is None:
            raise ValueError("target_disease missing")
        return self.disease_heads[self.target_disease](feat)

class EHRTokenizer:
    def __init__(self, max_len=512):
        self.max_len = max_len
        self.vocab = {"[PAD]": 0, "[CLS]": 1, "[UNK]": 2}
        self.pad_id = 0
        self.cls_id = 1
        self.unk_id = 2

    def encode(self, concepts, times):
        cids = [self.vocab.get(str(c), self.unk_id) for c in concepts]
        cids = [self.cls_id] + cids
        tids = [0] + list(times)

        if len(cids) > self.max_len:
            cids = cids[-self.max_len:]
            tids = tids[-self.max_len:]

        mask = [1] * len(cids)
        pad = self.max_len - len(cids)
        if pad > 0:
            cids += [self.pad_id] * pad
            tids += [0] * pad
            mask += [0] * pad

        return {
            "input_ids": torch.tensor(cids, dtype=torch.long),
            "time_ids": torch.tensor(tids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
        }


In [ ]:
import os
import copy
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.serialization import load as _native_torch_load

torch.load = _native_torch_load

# -------------------------
# 0) Config
# -------------------------
TEAM_BASE = "/home/jupyter/workspaces/controlled/cohort_definitions_teammate"
STATS_BASE = "/home/jupyter/workspaces/controlled/cohort_definitions"
CKPT_PATH = "/home/jupyter/workspaces/controlled/cohort_definitions_teammate/ehr_pretrained_ckpt_teammate_1774848487.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("TEAM_BASE:", TEAM_BASE)
print("STATS_BASE:", STATS_BASE)
print("CKPT:", CKPT_PATH)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------------
# 1) Required model objects
# -------------------------
for n in ["EHRTransformerEncoder", "MultiDiseaseClassifier", "EHRTokenizer"]:
    if n not in globals():
        raise RuntimeError(f"Missing required object in notebook: {n}")

# -------------------------
# 2) Compatibility patch (only if missing)
# -------------------------
def _safe_int_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        out = []
        for v in x:
            try:
                out.append(int(v))
            except Exception:
                out.append(0)
        return out
    return []

def _tokenize_one(tokenizer, concept_ids, time_deltas):
    c = [str(v) for v in concept_ids] if isinstance(concept_ids, (list, tuple, np.ndarray)) else []
    t = _safe_int_list(time_deltas)

    if hasattr(tokenizer, "encode"):
        try:
            out = tokenizer.encode(c, t)
            if isinstance(out, dict):
                ids = out.get("input_ids") or out.get("concept_ids")
                td  = out.get("time_deltas") or out.get("time_ids")
                am  = out.get("attention_mask")
                if ids is not None and td is not None and am is not None:
                    return list(ids), list(td), list(am)
            if isinstance(out, (tuple, list)) and len(out) >= 3:
                return list(out[0]), list(out[1]), list(out[2])
        except Exception:
            pass

    if callable(tokenizer):
        try:
            out = tokenizer(c, t)
            if isinstance(out, dict):
                ids = out.get("input_ids") or out.get("concept_ids")
                td  = out.get("time_deltas") or out.get("time_ids")
                am  = out.get("attention_mask")
                if ids is not None and td is not None and am is not None:
                    return list(ids), list(td), list(am)
        except Exception:
            pass

    vocab = getattr(tokenizer, "vocab", {})
    max_len = int(getattr(tokenizer, "max_len", 512))
    pad_id = int(vocab.get("[PAD]", 0))
    unk_id = int(vocab.get("[UNK]", 1))

    ids = [int(vocab.get(tok, unk_id)) for tok in c]
    td = t[:]
    if len(ids) > max_len:
        ids = ids[-max_len:]
        td = td[-max_len:]
    am = [1] * len(ids)
    if len(ids) < max_len:
        k = max_len - len(ids)
        ids += [pad_id] * k
        td += [0] * k
        am += [0] * k
    return ids, td, am

if "EHRDataset" not in globals():
    class EHRDataset(Dataset):
        def __init__(self, df, y, tokenizer):
            self.df = df.reset_index(drop=True)
            self.y = np.asarray(y).astype(np.float32)
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            r = self.df.iloc[idx]
            concept_ids = r["concept_ids"] if "concept_ids" in self.df.columns else []
            time_deltas = r["time_deltas"] if "time_deltas" in self.df.columns else []
            ids, td, am = _tokenize_one(self.tokenizer, concept_ids, time_deltas)
            return {
                "input_ids": torch.tensor(ids, dtype=torch.long),
                "time_deltas": torch.tensor(td, dtype=torch.long),
                "attention_mask": torch.tensor(am, dtype=torch.long),
                "labels": torch.tensor(self.y[idx], dtype=torch.float32),
            }

def _model_forward_fallback(model, batch):
    x = batch["input_ids"]
    t = batch["time_deltas"]
    m = batch["attention_mask"]
    tries = [
        lambda: model(input_ids=x, time_deltas=t, attention_mask=m),
        lambda: model(input_ids=x, time_ids=t, attention_mask=m),
        lambda: model(x, t, m),
        lambda: model(input_ids=x, attention_mask=m),
        lambda: model(x, m),
    ]
    last_e = None
    for fn in tries:
        try:
            return fn()
        except Exception as e:
            last_e = e
    raise RuntimeError(f"Model forward failed. Last error: {last_e}")

if "get_pytorch_predictions" not in globals():
    @torch.no_grad()
    def get_pytorch_predictions(model, loader, device="cuda"):
        model.eval()
        all_p = []
        for batch in loader:
            for k in ["input_ids", "time_deltas", "attention_mask"]:
                if k in batch:
                    batch[k] = batch[k].to(device)
            out = _model_forward_fallback(model, batch)
            if isinstance(out, (tuple, list)):
                out = out[0]
            if isinstance(out, dict):
                for kk in ["logits", "y_logit", "output", "pred"]:
                    if kk in out:
                        out = out[kk]
                        break
            out = torch.as_tensor(out)
            if out.ndim > 1:
                out = out.squeeze(-1)
            vmin = float(out.min().detach().cpu())
            vmax = float(out.max().detach().cpu())
            prob = out if (vmin >= 0 and vmax <= 1) else torch.sigmoid(out)
            all_p.append(prob.detach().cpu().numpy())
        return np.concatenate(all_p).astype(float) if len(all_p) else np.array([], dtype=float)

if "generate_patient_level_bootstrap_indices_binary" not in globals():
    def generate_patient_level_bootstrap_indices_binary(patient_ids, y_true, n_bootstrap=200, random_state=42):
        rng = np.random.RandomState(random_state)
        patient_ids = np.asarray(patient_ids)
        uniq = np.unique(patient_ids)
        pid_to_idx = {p: np.where(patient_ids == p)[0] for p in uniq}
        out = []
        for _ in range(n_bootstrap):
            samp = rng.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([pid_to_idx[p] for p in samp])
            out.append(idx)
        return out

if "evaluate_predictions_on_bootstrap_indices" not in globals():
    def evaluate_predictions_on_bootstrap_indices(y_true, y_pred, boot_idx):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(float)
        auroc_sc, auprc_sc = [], []
        for idx in boot_idx:
            yt = y_true[idx]
            yp = y_pred[idx]
            if len(np.unique(yt)) < 2:
                auroc_sc.append(np.nan)
                auprc_sc.append(np.nan)
                continue
            auroc_sc.append(roc_auc_score(yt, yp))
            auprc_sc.append(average_precision_score(yt, yp))
        return np.array(auroc_sc, dtype=float), np.array(auprc_sc, dtype=float)

# -------------------------
# 3) IO helpers
# -------------------------
def load_ehr_disease_data_from_base(endpoint, base_path):
    full = f"{base_path}/{endpoint}"
    split_p = f"{full}/patient_splits/master_shared_split_90d.parquet"
    label_p = f"{full}/parquet_labels/label_{endpoint}_ehr.parquet"
    seq_p   = f"{full}/inspectomop_features/transformer_{endpoint}_ehr.parquet"
    for p in [split_p, label_p, seq_p]:
        if not os.path.exists(p):
            raise FileNotFoundError(p)

    splits = pd.read_parquet(split_p)
    labels = pd.read_parquet(label_p)
    seqs   = pd.read_parquet(seq_p)

    if "subject_id" in labels.columns and "person_id" not in labels.columns:
        labels = labels.rename(columns={"subject_id": "person_id"})
    if "prediction_time" in labels.columns and "index_date" not in labels.columns:
        labels = labels.rename(columns={"prediction_time": "index_date"})
    if "boolean_value" not in labels.columns:
        for c in ["label", "target", "outcome", "boolean", "y"]:
            if c in labels.columns:
                labels = labels.rename(columns={c: "boolean_value"})
                break

    labels["person_id"] = pd.to_numeric(labels["person_id"], errors="coerce")
    labels["index_date"] = pd.to_datetime(labels["index_date"], errors="coerce")
    labels["boolean_value"] = pd.to_numeric(labels["boolean_value"], errors="coerce").fillna(0).astype(int).clip(0, 1)
    seqs["person_id"] = pd.to_numeric(seqs["person_id"], errors="coerce")
    seqs["index_date"] = pd.to_datetime(seqs["index_date"], errors="coerce")

    labels = labels.dropna(subset=["person_id","index_date"]).copy()
    seqs = seqs.dropna(subset=["person_id","index_date"]).copy()
    labels["person_id"] = labels["person_id"].astype("int64")
    seqs["person_id"] = seqs["person_id"].astype("int64")

    master = labels.merge(seqs, on=["person_id","index_date"], how="inner")

    train_ids = set(pd.to_numeric(splits[splits["split"]=="train"]["person_id"], errors="coerce").dropna().astype("int64"))
    val_ids   = set(pd.to_numeric(splits[splits["split"]=="val"]["person_id"], errors="coerce").dropna().astype("int64"))
    test_ids  = set(pd.to_numeric(splits[splits["split"]=="test"]["person_id"], errors="coerce").dropna().astype("int64"))

    tr = master[master["person_id"].isin(train_ids)].sort_values(["person_id","index_date"]).reset_index(drop=True)
    va = master[master["person_id"].isin(val_ids)].sort_values(["person_id","index_date"]).reset_index(drop=True)
    te = master[master["person_id"].isin(test_ids)].sort_values(["person_id","index_date"]).reset_index(drop=True)
    return tr, va, te

# -------------------------
# 4) Anti-leak sanitize
# -------------------------
LEAK_EXACT = {
    "boolean_value", "label", "target", "outcome", "y",
    "split", "fold", "set", "is_train", "is_val", "is_test"
}
LEAK_PAT = re.compile(r"(label|target|outcome|boolean|^y$|pred|prediction|future|post|split|fold)", re.I)

def sanitize_wearable_stats(stats_df: pd.DataFrame):
    x = stats_df.copy()
    if "subject_id" in x.columns and "person_id" not in x.columns:
        x = x.rename(columns={"subject_id": "person_id"})
    if "prediction_time" in x.columns and "index_date" not in x.columns:
        x = x.rename(columns={"prediction_time": "index_date"})

    if "person_id" not in x.columns or "index_date" not in x.columns:
        raise ValueError("stats missing person_id/index_date")

    x["person_id"] = pd.to_numeric(x["person_id"], errors="coerce")
    x["index_date"] = pd.to_datetime(x["index_date"], errors="coerce")
    x = x.dropna(subset=["person_id","index_date"]).copy()
    x["person_id"] = x["person_id"].astype("int64")

    drop_cols = [c for c in x.columns if (c in LEAK_EXACT) or LEAK_PAT.search(c)]
    drop_cols = [c for c in drop_cols if c not in ["person_id","index_date"]]
    x = x.drop(columns=drop_cols, errors="ignore")

    feat_cols = [c for c in x.columns if c not in ["person_id","index_date"] and pd.api.types.is_numeric_dtype(x[c])]
    x = x[["person_id","index_date"] + feat_cols].copy()

    bad_left = [c for c in x.columns if c not in ["person_id","index_date"] and ((c in LEAK_EXACT) or LEAK_PAT.search(c))]
    if bad_left:
        raise ValueError(f"Leak-like columns remain after sanitize: {bad_left}")

    if len(feat_cols) > 0:
        x = x.groupby(["person_id","index_date"], as_index=False)[feat_cols].mean()
    else:
        x = x.drop_duplicates(subset=["person_id","index_date"]).copy()
    return x, drop_cols, feat_cols

def load_clean_stats(disease):
    p = f"{STATS_BASE}/{disease}/inspectomop_features/transformer_{disease}_multimodal_stats.parquet"
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    raw = pd.read_parquet(p)
    clean, dropped, feats = sanitize_wearable_stats(raw)
    return clean, dropped, feats, p

# -------------------------
# 5) Build pretrained EHR
# -------------------------
def build_ehr_from_ckpt(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    vocab = sd["encoder.concept_embedding.weight"].shape[0]
    hidden = sd["encoder.concept_embedding.weight"].shape[1]
    heads = sorted({k.split(".")[1] for k in sd if k.startswith("disease_heads.") and k.endswith(".0.weight")})

    ehr = MultiDiseaseClassifier(
        shared_encoder=EHRTransformerEncoder(vocab_size=vocab, max_days=1095, hidden_dim=hidden, n_layers=4, n_heads=8),
        disease_list=heads,
        encoder_out_dim=hidden,
        mlp_hidden_dim=64
    ).to(device)
    ehr.load_state_dict(sd)
    ehr.eval()

    tk = EHRTokenizer(max_len=ckpt.get("tokenizer_max_len", 512))
    tk.vocab = ckpt["tokenizer_vocab"]

    print(f"ckpt loaded | vocab={vocab}, hidden={hidden}, diseases={len(heads)}")
    return ehr, tk

BASE_EHR_MODEL, TOKENIZER = build_ehr_from_ckpt(CKPT_PATH, DEVICE)

# -------------------------
# 6) Model and metrics
# -------------------------
class WearResidualNet(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.2):
        super().__init__()
        if in_dim <= 0:
            self.net = None
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.ReLU(),
                nn.Dropout(drop),
                nn.Linear(hidden, 1)
            )

    def forward(self, x):
        if self.net is None:
            return torch.zeros((x.shape[0], 1), device=x.device, dtype=x.dtype)
        return self.net(x)

def safe_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def safe_auc(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def get_ehr_probs_for_df(ehr_model, tokenizer, disease, df, y, device, batch_size=128):
    ds = EHRDataset(df, y, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    ehr_model.set_target_disease(disease)
    p = get_pytorch_predictions(ehr_model, loader, device=device)
    return np.asarray(p).astype(float)

def make_wear_arrays(df, wear_cols, mu=None, sigma=None):
    if len(wear_cols) == 0:
        n = len(df)
        X = np.zeros((n, 1), dtype=np.float32)
        mask = np.zeros(n, dtype=np.float32)
        if mu is None:
            mu = np.zeros(1, dtype=np.float32)
            sigma = np.ones(1, dtype=np.float32)
        return X, mask, mu, sigma

    W = df[wear_cols].copy()
    mask = W.notna().any(axis=1).astype(float).values.astype(np.float32)
    W = W.astype(float)

    if mu is None or sigma is None:
        mu = W.mean(axis=0).values
        sigma = W.std(axis=0).replace(0, 1).fillna(1).values

    W = (W - mu) / sigma
    W = W.fillna(0.0)
    X = W.values.astype(np.float32)
    return X, mask, mu.astype(np.float32), sigma.astype(np.float32)

def bootstrap_metrics(y_true, y_prob, pids, n_boot=200):
    idx = generate_patient_level_bootstrap_indices_binary(
        pids.astype("int64"), y_true.astype(int), n_bootstrap=n_boot, random_state=42
    )
    auroc_sc, auprc_sc = evaluate_predictions_on_bootstrap_indices(y_true.astype(int), y_prob.astype(float), idx)
    auroc = auroc_sc[~np.isnan(auroc_sc)]
    auprc = auprc_sc[~np.isnan(auprc_sc)]
    return (
        float(np.mean(auroc)) if len(auroc) else np.nan,
        float(np.std(auroc, ddof=1)) if len(auroc) > 1 else np.nan,
        float(np.mean(auprc)) if len(auprc) else np.nan,
        float(np.std(auprc, ddof=1)) if len(auprc) > 1 else np.nan,
    )

def alpha_for_epoch(ep):
    if ep <= 10:
        return 0.0, "ehr"
    if ep <= 20:
        return 0.015 * (ep - 10), "wear"
    return 0.155 + 0.005 * (ep - 21), "joint"


In [ ]:
# QC residual fusion trainer（支持任意病种；后续用于10病批量训练）

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def sanitize_qc_stats(df):
    x = df.copy()
    if "subject_id" in x.columns and "person_id" not in x.columns:
        x = x.rename(columns={"subject_id":"person_id"})
    if "prediction_time" in x.columns and "index_date" not in x.columns:
        x = x.rename(columns={"prediction_time":"index_date"})
    x["person_id"] = pd.to_numeric(x["person_id"], errors="coerce")
    x["index_date"] = pd.to_datetime(x["index_date"], errors="coerce")
    x = x.dropna(subset=["person_id","index_date"]).copy()
    x["person_id"] = x["person_id"].astype("int64")

    bad_pat = r"(label|target|outcome|boolean|^y$|split|fold|pred|prediction)"
    drop_cols = [c for c in x.columns if pd.Series([c]).str.contains(bad_pat, case=False, regex=True).iloc[0]]
    drop_cols = [c for c in drop_cols if c not in ["person_id","index_date"]]
    x = x.drop(columns=drop_cols, errors="ignore")

    feat_cols = [c for c in x.columns if c not in ["person_id","index_date"] and pd.api.types.is_numeric_dtype(x[c])]
    x = x[["person_id","index_date"] + feat_cols].copy()
    x = x.groupby(["person_id","index_date"], as_index=False)[feat_cols].mean()
    return x, feat_cols

class WearResidualNet(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

def alpha_for_epoch(ep):
    if ep <= 10:
        return 0.0, "ehr"
    if ep <= 20:
        return 0.015 * (ep - 10), "wear"
    return 0.155 + 0.005 * (ep - 21), "joint"

def quality_weight(df):
    n = df["wear_n_days_any"].fillna(0).values.astype(np.float32) if "wear_n_days_any" in df.columns else np.zeros(len(df), dtype=np.float32)
    g = df["wear_gap_days"].fillna(365).clip(0,365).values.astype(np.float32) if "wear_gap_days" in df.columns else np.ones(len(df), dtype=np.float32)*365
    r = df["wear_recent30_valid_ratio"].fillna(0).clip(0,1).values.astype(np.float32) if "wear_recent30_valid_ratio" in df.columns else np.zeros(len(df), dtype=np.float32)

    q1 = np.clip(n / 90.0, 0, 1)
    q2 = np.exp(-g / 120.0)
    q3 = 0.5 + 0.5 * r
    q = np.clip(q1 * q2 * q3, 0, 1).astype(np.float32)
    return q

def make_arrays(df, wear_cols, mu=None, sigma=None):
    W = df[wear_cols].copy()
    m = W.notna().any(axis=1).astype(np.float32).values
    W = W.astype(float)
    if mu is None or sigma is None:
        mu = W.mean(axis=0).values
        sigma = W.std(axis=0).replace(0,1).fillna(1).values
    W = (W - mu) / sigma
    W = W.fillna(0.0)
    return W.values.astype(np.float32), m, mu.astype(np.float32), sigma.astype(np.float32)

def run_one_disease_qc(disease, max_epochs=30, batch_size=128, patience=6):
    tr, va, te = load_ehr_disease_data_from_base(disease, TEAM_BASE)

    p = f"{STATS_BASE}/{disease}/inspectomop_features/transformer_{disease}_multimodal_stats_qc180_local.parquet"
    st_raw = pd.read_parquet(p)
    st, wear_cols = sanitize_qc_stats(st_raw)

    trm = tr.merge(st, on=["person_id","index_date"], how="left")
    vam = va.merge(st, on=["person_id","index_date"], how="left")
    tem = te.merge(st, on=["person_id","index_date"], how="left")

    y_tr = trm["boolean_value"].astype(int).values
    y_va = vam["boolean_value"].astype(int).values
    y_te = tem["boolean_value"].astype(int).values
    pid_te = tem["person_id"].astype("int64").values

    p_ehr_tr = get_ehr_probs_for_df(BASE_EHR_MODEL, TOKENIZER, disease, trm, y_tr, DEVICE, batch_size=batch_size)
    p_ehr_va = get_ehr_probs_for_df(BASE_EHR_MODEL, TOKENIZER, disease, vam, y_va, DEVICE, batch_size=batch_size)
    p_ehr_te = get_ehr_probs_for_df(BASE_EHR_MODEL, TOKENIZER, disease, tem, y_te, DEVICE, batch_size=batch_size)

    l_tr = safe_logit(p_ehr_tr).astype(np.float32)
    l_va = safe_logit(p_ehr_va).astype(np.float32)
    l_te = safe_logit(p_ehr_te).astype(np.float32)

    X_tr, m_tr, mu, sigma = make_arrays(trm, wear_cols, None, None)
    X_va, m_va, _, _ = make_arrays(vam, wear_cols, mu, sigma)
    X_te, m_te, _, _ = make_arrays(tem, wear_cols, mu, sigma)

    q_tr = quality_weight(trm)
    q_va = quality_weight(vam)
    q_te = quality_weight(tem)

    ds = TensorDataset(
        torch.from_numpy(X_tr), torch.from_numpy(m_tr), torch.from_numpy(q_tr),
        torch.from_numpy(l_tr), torch.from_numpy(y_tr.astype(np.float32))
    )
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = WearResidualNet(X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss()

    X_va_t = torch.from_numpy(X_va).to(DEVICE)
    m_va_t = torch.from_numpy(m_va).to(DEVICE)
    q_va_t = torch.from_numpy(q_va).to(DEVICE)
    l_va_t = torch.from_numpy(l_va).to(DEVICE)

    best_state = copy.deepcopy(model.state_dict())
    best_val = -np.inf
    best_alpha = 0.0
    bad = 0

    print(f"\n=== {disease} ===")
    print(f"rows tr/va/te={len(trm)}/{len(vam)}/{len(tem)} | wear cols={len(wear_cols)} | cover test={(m_te>0).mean():.3f}")

    for ep in range(1, max_epochs+1):
        alpha, phase = alpha_for_epoch(ep)
        model.train()
        loss_sum, n_sum = 0.0, 0

        for xb, mb, qb, lb, yb in dl:
            xb, mb, qb, lb, yb = xb.to(DEVICE), mb.to(DEVICE), qb.to(DEVICE), lb.to(DEVICE), yb.to(DEVICE)

            if phase == "ehr":
                with torch.no_grad():
                    wl = model(xb).squeeze(1)
                    fused = lb + alpha * mb * qb * wl
                    loss = bce(fused, yb)
            else:
                opt.zero_grad()
                wl = model(xb).squeeze(1)
                fused = lb + alpha * mb * qb * wl
                loss = bce(fused, yb)
                loss.backward()
                opt.step()

            loss_sum += float(loss.item()) * len(xb)
            n_sum += len(xb)

        model.eval()
        with torch.no_grad():
            wl_va = model(X_va_t).squeeze(1)
            p_fus_va = torch.sigmoid(l_va_t + alpha * m_va_t * q_va_t * wl_va).cpu().numpy()

        val_fus = safe_auc(y_va, p_fus_va)
        val_ehr = safe_auc(y_va, p_ehr_va)
        print(f"epoch {ep:02d} | {phase:<5} alpha={alpha:.3f} | loss={loss_sum/max(n_sum,1):.4f} | val_ehr={val_ehr:.4f} | val_fus={val_fus:.4f}")

        if ep >= 11:
            if np.isfinite(val_fus) and val_fus > best_val + 1e-5:
                best_val = val_fus
                best_alpha = alpha
                best_state = copy.deepcopy(model.state_dict())
                bad = 0
            else:
                bad += 1
            if bad >= patience:
                print(f"early stop | best_val_fus={best_val:.4f} @ alpha={best_alpha:.3f}")
                break

    model.load_state_dict(best_state)
    model.eval()
    X_te_t = torch.from_numpy(X_te).to(DEVICE)
    m_te_t = torch.from_numpy(m_te).to(DEVICE)
    q_te_t = torch.from_numpy(q_te).to(DEVICE)
    l_te_t = torch.from_numpy(l_te).to(DEVICE)

    with torch.no_grad():
        wl_te = model(X_te_t).squeeze(1)
        p_fus_te = torch.sigmoid(l_te_t + best_alpha * m_te_t * q_te_t * wl_te).cpu().numpy()

    ehr_auroc_mean, ehr_auroc_std, ehr_auprc_mean, ehr_auprc_std = bootstrap_metrics(y_te, p_ehr_te, pid_te, n_boot=200)
    fus_auroc_mean, fus_auroc_std, fus_auprc_mean, fus_auprc_std = bootstrap_metrics(y_te, p_fus_te, pid_te, n_boot=200)

    return {
        "disease": disease,
        "n_test": len(tem),
        "wear_test_coverage": float((m_te>0).mean()),
        "n_wear_feat": len(wear_cols),
        "best_alpha": float(best_alpha),
        "ehr_only_auroc_mean": ehr_auroc_mean,
        "fusion_auroc_mean": fus_auroc_mean,
        "delta_fusion_minus_ehr": float(fus_auroc_mean - ehr_auroc_mean),
        "ehr_only_auprc_mean": ehr_auprc_mean,
        "fusion_auprc_mean": fus_auprc_mean,
    }


In [ ]:
# Cell 1: 补齐 10 病种 QC180 wearable snapshot 特征（同一数据质量工程）
import os
import duckdb
import numpy as np
import pandas as pd

TEAM_BASE = "/home/jupyter/workspaces/controlled/cohort_definitions_teammate"
STATS_BASE = "/home/jupyter/workspaces/controlled/cohort_definitions"
WEAR_PATH = "/home/jupyter/workspaces/controlled/wearable_daily_merged.parquet"

OUTCOMES = [
    "hyperlipidemia","hypertension","gerd","gad","mdd",
    "obesity","sleep_apnea","type2_diabetes","hf","atrial_fibrillation"
]
WINDOW_DAYS = 180
FORCE_REBUILD = False  # True=全部重建；False=只补缺失

os.makedirs("/home/jupyter/workspaces/controlled/report", exist_ok=True)

def load_ehr_anchor(d):
    p = f"{TEAM_BASE}/{d}/parquet_labels/label_{d}_ehr.parquet"
    x = pd.read_parquet(p).copy()
    if "subject_id" in x.columns and "person_id" not in x.columns:
        x = x.rename(columns={"subject_id":"person_id"})
    if "prediction_time" in x.columns and "index_date" not in x.columns:
        x = x.rename(columns={"prediction_time":"index_date"})
    x["person_id"] = pd.to_numeric(x["person_id"], errors="coerce")
    x["index_date"] = pd.to_datetime(x["index_date"], errors="coerce").dt.date
    x = x.dropna(subset=["person_id","index_date"]).copy()
    x["person_id"] = x["person_id"].astype("int64")
    return x[["person_id","index_date"]].drop_duplicates()

con = duckdb.connect()

build_rows = []
for d in OUTCOMES:
    out_p = f"{STATS_BASE}/{d}/inspectomop_features/transformer_{d}_multimodal_stats_qc180_local.parquet"
    if (not FORCE_REBUILD) and os.path.exists(out_p):
        df0 = pd.read_parquet(out_p, columns=["person_id","index_date","wear_n_days_any"])
        build_rows.append({
            "disease": d,
            "status": "skip_exists",
            "n_rows": len(df0),
            "coverage_rate": float((df0["wear_n_days_any"] > 0).mean()),
            "out_path": out_p
        })
        print(f"[{d}] skip (exists) -> {out_p}")
        continue

    anchors = load_ehr_anchor(d)
    con.register("anchors_df", anchors)

    q = f"""
    WITH anchors AS (
      SELECT CAST(person_id AS BIGINT) AS person_id, CAST(index_date AS DATE) AS index_date
      FROM anchors_df
    ),
    wear AS (
      SELECT
        CAST(person_id AS BIGINT) AS person_id,
        CAST(day AS DATE) AS day,
        LEAST(GREATEST(steps_day, 0), 150000) AS steps_day,
        LEAST(GREATEST(hr_avg_day, 30), 220) AS hr_avg_day,
        LEAST(GREATEST(hr_min_day, 20), 180) AS hr_min_day,
        LEAST(GREATEST(hr_max_day, 30), 260) AS hr_max_day,
        LEAST(GREATEST(hr_n, 0), 2000) AS hr_n,
        LEAST(GREATEST(sleep_total_day, 0), 1440) AS sleep_total_day,
        LEAST(GREATEST(sleep_deep_day, 0), 900) AS sleep_deep_day,
        LEAST(GREATEST(sleep_rem_day, 0), 900) AS sleep_rem_day,
        LEAST(GREATEST(activity_calories_day, 0), 12000) AS activity_calories_day,
        LEAST(GREATEST(activity_steps_day, 0), 150000) AS activity_steps_day
      FROM read_parquet('{WEAR_PATH}')
    ),
    j AS (
      SELECT
        a.person_id, a.index_date, w.day,
        w.steps_day, w.hr_avg_day, w.hr_min_day, w.hr_max_day, w.hr_n,
        w.sleep_total_day, w.sleep_deep_day, w.sleep_rem_day,
        w.activity_calories_day, w.activity_steps_day
      FROM anchors a
      LEFT JOIN wear w
        ON w.person_id = a.person_id
       AND w.day BETWEEN a.index_date - INTERVAL {WINDOW_DAYS} DAY AND a.index_date - INTERVAL 1 DAY
    )
    SELECT
      person_id,
      index_date,
      COUNT(day) AS wear_n_days_any,
      DATE_DIFF('day', MAX(day), index_date) AS wear_gap_days,
      AVG(CASE WHEN day >= index_date - INTERVAL 30 DAY THEN 1 ELSE 0 END) AS wear_recent30_valid_ratio,

      AVG(steps_day) AS wear_steps_day_mean,
      STDDEV_SAMP(steps_day) AS wear_steps_day_std,
      MEDIAN(steps_day) AS wear_steps_day_med,
      AVG(CASE WHEN day >= index_date - INTERVAL 7 DAY THEN steps_day END) AS wear_steps_day_mean_7d,
      AVG(CASE WHEN day >= index_date - INTERVAL 30 DAY THEN steps_day END) AS wear_steps_day_mean_30d,

      AVG(hr_avg_day) AS wear_hr_avg_day_mean,
      STDDEV_SAMP(hr_avg_day) AS wear_hr_avg_day_std,
      MEDIAN(hr_avg_day) AS wear_hr_avg_day_med,
      AVG(CASE WHEN day >= index_date - INTERVAL 7 DAY THEN hr_avg_day END) AS wear_hr_avg_day_mean_7d,
      AVG(CASE WHEN day >= index_date - INTERVAL 30 DAY THEN hr_avg_day END) AS wear_hr_avg_day_mean_30d,

      AVG(hr_min_day) AS wear_hr_min_day_mean,
      AVG(hr_max_day) AS wear_hr_max_day_mean,
      AVG(hr_n) AS wear_hr_n_mean,

      AVG(sleep_total_day) AS wear_sleep_total_day_mean,
      STDDEV_SAMP(sleep_total_day) AS wear_sleep_total_day_std,
      AVG(CASE WHEN day >= index_date - INTERVAL 7 DAY THEN sleep_total_day END) AS wear_sleep_total_day_mean_7d,
      AVG(CASE WHEN day >= index_date - INTERVAL 30 DAY THEN sleep_total_day END) AS wear_sleep_total_day_mean_30d,

      AVG(sleep_deep_day) AS wear_sleep_deep_day_mean,
      AVG(sleep_rem_day) AS wear_sleep_rem_day_mean,

      AVG(activity_calories_day) AS wear_activity_calories_day_mean,
      AVG(activity_steps_day) AS wear_activity_steps_day_mean
    FROM j
    GROUP BY person_id, index_date
    """

    feat = con.execute(q).df()
    os.makedirs(os.path.dirname(out_p), exist_ok=True)
    feat.to_parquet(out_p, index=False)

    build_rows.append({
        "disease": d,
        "status": "rebuilt",
        "n_rows": len(feat),
        "coverage_rate": float((feat["wear_n_days_any"] > 0).mean()),
        "out_path": out_p
    })
    print(f"[{d}] rebuilt -> rows={len(feat)} cov={(feat['wear_n_days_any']>0).mean():.3%}")

build_df = pd.DataFrame(build_rows).sort_values("disease")
display(build_df)
build_df.to_csv("/home/jupyter/workspaces/controlled/report/qc180_build_10d_summary.csv", index=False)
print("saved: /home/jupyter/workspaces/controlled/report/qc180_build_10d_summary.csv")


In [ ]:
# Cell 2: 训练前快速泄漏检查（应全部通过）
import re
import pandas as pd

pat = re.compile(r"(label|target|outcome|boolean|^y$|split|fold|pred|prediction)", re.I)

chk_rows = []
for d in OUTCOMES:
    p = f"{STATS_BASE}/{d}/inspectomop_features/transformer_{d}_multimodal_stats_qc180_local.parquet"
    df = pd.read_parquet(p)
    bad = [c for c in df.columns if pat.search(c)]
    chk_rows.append({
        "disease": d,
        "n_rows": len(df),
        "n_cols": df.shape[1],
        "bad_cols_n": len(bad),
        "coverage_rate": float((df["wear_n_days_any"] > 0).mean()),
        "bad_cols": "|".join(bad[:5])
    })

chk = pd.DataFrame(chk_rows).sort_values("disease")
display(chk)
assert (chk["bad_cols_n"] == 0).all(), "发现可疑泄漏列，请先处理"
print("Leakage check passed.")


In [ ]:
# Cell 3: 10病训练 + 总结表（复用你已有 run_one_disease_qc）
# 要求：run_one_disease_qc() 已定义，并且内部读取 *_multimodal_stats_qc180_local.parquet

OUT_CSV_10D = "/home/jupyter/workspaces/controlled/report/ehr_residual_fusion_qc180_local_10d.csv"
OUT_SUMMARY = "/home/jupyter/workspaces/controlled/report/ehr_residual_fusion_qc180_local_10d_summary.csv"

rows = []
for i, d in enumerate(OUTCOMES, 1):
    print(f"\n[{i}/{len(OUTCOMES)}] {d}")
    rows.append(run_one_disease_qc(d, max_epochs=30, batch_size=128, patience=6))

res10 = pd.DataFrame(rows).sort_values("disease").reset_index(drop=True)
res10.to_csv(OUT_CSV_10D, index=False)

summary_cols = [
    "disease",
    "n_test",
    "wear_test_coverage",
    "n_wear_feat",
    "best_alpha",
    "ehr_only_auroc_mean",
    "fusion_auroc_mean",
    "delta_fusion_minus_ehr",
    "ehr_only_auprc_mean",
    "fusion_auprc_mean",
]
summary = res10[summary_cols].copy().sort_values("delta_fusion_minus_ehr", ascending=False).reset_index(drop=True)

macro_row = pd.DataFrame([{
    "disease": "MACRO_MEAN",
    "n_test": int(res10["n_test"].sum()),
    "wear_test_coverage": float(res10["wear_test_coverage"].mean()),
    "n_wear_feat": float(res10["n_wear_feat"].mean()),
    "best_alpha": float(res10["best_alpha"].mean()),
    "ehr_only_auroc_mean": float(res10["ehr_only_auroc_mean"].mean()),
    "fusion_auroc_mean": float(res10["fusion_auroc_mean"].mean()),
    "delta_fusion_minus_ehr": float((res10["fusion_auroc_mean"] - res10["ehr_only_auroc_mean"]).mean()),
    "ehr_only_auprc_mean": float(res10["ehr_only_auprc_mean"].mean()),
    "fusion_auprc_mean": float(res10["fusion_auprc_mean"].mean()),
}])

summary_all = pd.concat([summary, macro_row], ignore_index=True)
display(summary_all)

summary_all.to_csv(OUT_SUMMARY, index=False)
print("Saved full:", OUT_CSV_10D)
print("Saved summary:", OUT_SUMMARY)
